In [1]:
import joblib
import numpy as np
import dashscope
import json
import warnings
warnings.filterwarnings("ignore", category=UserWarning)


# ========== 0. 全局初始化（只做一次） ==========
dashscope.api_key = ""

model = joblib.load("diabetes_model.pkl")
scaler = joblib.load("scaler.pkl")

FEATURE_NAMES = [
    "怀孕次数", "血糖浓度", "舒张压", "三头肌皮褶厚度",
    "2小时血清胰岛素", "BMI", "糖尿病遗传函数", "年龄"
]


# ========== 1. 预测函数（黑盒子） ==========
def predict_patient(patient_dict):
    """
    输入：patient_dict（字典，8个指标的中文名做key，数值做value）
    输出：result（字典，包含 risk_level、main_factors、suggestions、disclaimer、prob）
    """
    # 1.1 把字典按顺序转成列表 → 二维数组
    patient_data = [patient_dict[name] for name in FEATURE_NAMES]
    patient_array = np.array([patient_data])
    
    # 1.2 标准化 + 预测概率
    patient_scaled = scaler.transform(patient_array)
    risk_prob = model.predict_proba(patient_scaled)[0, 1]
    
    # 1.3 调用 Qwen
    prompt = f"""
你是一名内分泌临床医师。请根据患者指标和模型预测概率，输出一段 JSON 格式的评估结果。
患者指标：{patient_dict}
模型预测患病概率：{risk_prob:.2%}

请严格按照以下 JSON 结构输出，不要包含任何其他文字：
{{
    "risk_level": "高风险/中风险/低风险",
    "main_factors": ["因素1", "因素2"],
    "suggestions": ["建议1", "建议2"],
    "disclaimer": "本结果仅供参考，不能替代临床就诊。"
}}
"""
    resp = dashscope.Generation.call(
        model="qwen-max",
        messages=[{"role": "user", "content": prompt}]
    )
    
    # 1.4 解析
    if resp.status_code == 200:
        try:
            result = json.loads(resp.output.text)
        except json.JSONDecodeError:
            result = {
                "risk_level": "解析失败",
                "main_factors": [],
                "suggestions": ["模型未按要求输出JSON，请检查Prompt"],
                "disclaimer": "本结果仅供参考"
            }
    else:
        result = {
            "risk_level": "调用失败",
            "main_factors": [],
            "suggestions": [resp.message],
            "disclaimer": "本结果仅供参考"
        }
    
    # 1.5 把概率也塞进结果字典（方便以后使用）
    result["prob"] = risk_prob
    return result


# ========== 2. 打印函数（给人类看） ==========
def print_report(result):
    print("\n" + "=" * 40)
    print("      糖尿病风险评估报告")
    print("=" * 40)
    print(f"风险等级：{result['risk_level']}")
    print(f"模型概率：{result['prob']:.2%}")
    print(f"主要因素：{'、'.join(result['main_factors']) if result['main_factors'] else '无'}")
    print("生活建议：")
    for i, s in enumerate(result["suggestions"], 1):
        print(f"  {i}. {s}")
    print(f"\n免责声明：{result['disclaimer']}")
    print("=" * 40)

In [2]:
# 模拟一个患者
patient = {
    "怀孕次数": 2, "血糖浓度": 130, "舒张压": 75,
    "三头肌皮褶厚度": 35, "2小时血清胰岛素": 200,
    "BMI": 32, "糖尿病遗传函数": 0.5, "年龄": 45
}

result = predict_patient(patient)
print_report(result)


      糖尿病风险评估报告
风险等级：中风险
模型概率：39.96%
主要因素：血糖浓度、BMI
生活建议：
  1. 定期监测血糖水平
  2. 采取措施控制体重，如增加体育活动、调整饮食结构

免责声明：本结果仅供参考，不能替代临床就诊。
